<div align="center">

<a href="https://colab.research.google.com/github/utkukose/cyber-physical-systems-NB-lecture/blob/main/exams/final/FIN3_pumping_station.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final option 3: A pumping station with a safety case

**Cyber-Physical Systems (BIL-435 Siber Fiziksel Sistemler)**<br>Final project: A networked cyber-physical system with its digital twin. Starter Colab notebook

[Project page](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/exams/final) | [Course repository](https://github.com/utkukose/cyber-physical-systems-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A pumping station that fills a hillside reservoir over a radio link. Replace the values marked `# TODO` with the data of your own system. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. Telemetry over a radio link

*Lecture: [Week 8, Latency, queues and repetition](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#latency) and [Week 8, Control over a network](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#control-over-a-network)*

A pumping station in the valley fills the reservoir of a hillside campus. The level transmitter at the reservoir sends
a reading every $T_s$ seconds over a radio link, and each message is lost with the probability $p$. The station must not
pump blindly, so it stops the pump when $k$ readings in a row are missing. All $k$ readings are lost with the probability

$$P_{miss} = p^{k},$$

so about $N\,(1 - p)\,p^k$ false stops happen among $N$ readings, each after a received reading. A larger $k$ avoids
false stops but detects a real loss of the link only after $k\,T_s$ seconds. Meanwhile the pump keeps filling, and the
level rises by up to $Q\,k\,T_s/A$. The cell chooses the smallest $k$ with fewer false stops than the target and
computes the rise of the level until the pump stops.

In [ ]:
# TODO: the radio link and the reservoir of your system
T_S = 10.0                   # period of the readings (s)
P_LOST = 0.05                # probability that one message is lost
FALSE_STOPS_PER_YEAR = 1.0   # target for the false stops of the pump
A = 80.0                     # surface of the reservoir (m²)
Q = 60.0                     # flow of the pump (m³/h)

N_year = 365 * 86400 / T_S
rows = []
for k in range(1, 9):
    false_stops = N_year * (1 - P_LOST) * P_LOST ** k
    rows.append({"k": k, "P_miss": f"{P_LOST ** k:.1e}", "false stops per year": round(false_stops, 3),
                 "detection (s)": k * T_S, "rise until the stop (mm)": round(1000 * Q / 3600 * k * T_S / A, 2)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
K_LINK = int(table[table["false stops per year"] < FALSE_STOPS_PER_YEAR].k.iloc[0])
print(f"Chosen k = {K_LINK}: the station notices a lost link after {K_LINK * T_S:.0f} s")
RESULTS["readings missed before a stop (k)"] = K_LINK
RESULTS["detection of a lost link (s)"] = K_LINK * T_S
RESULTS["false stops per year"] = float(table.loc[table.k == K_LINK, "false stops per year"].iloc[0])

## 2. The twin of the reservoir

*Lecture: [Week 12, Calibration: Fitting the twin to the library](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#calibration)*

The twin of the reservoir is the tank model of Week 2, $A\,dh/dt = Q\,u - D(t)$, with the pump state $u$ of 0 or 1. The
flow $Q$ of the pump depends on its wear, and the demand $D$ of the campus is not measured. Over one step $\Delta t$ of
the log, the model gives a linear equation in the unknowns,

$$A\,\frac{h_{k+1} - h_k}{\Delta t} = Q\,u_k - D_{j(k)} ,$$

where $D_j$ is the demand in the hour $j$ of the day. Linear least squares finds $Q$ and the 24 hourly demands from all
steps of the log at once. The last day of the log validates the twin. Run alone over a whole day, the twin drifts,
because the reservoir integrates every small error of the demand. A synchronised twin starts each prediction from the
latest reading, so the validation measures its predictions one hour ahead.

In [ ]:
# TODO: replace the simulated log with the log of your station: one row every 10 minutes
DT_H = 1 / 6                 # step of the log (h)
DAYS = 5

def station_history(days=DAYS, seed=41):
    """A simulated log under the float switches: level with noise and the pump state."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(days * 24 / DT_H)) * DT_H
    hour = t % 24
    demand = 18 + 22 * np.exp(-0.5 * ((hour - 8) / 1.5) ** 2) + 16 * np.exp(-0.5 * ((hour - 19) / 2) ** 2)
    demand = demand * np.repeat(rng.normal(1, 0.08, days), int(24 / DT_H)) + rng.normal(0, 2, t.size)
    level, pump, rows = 3.0, 0, []
    for k in range(t.size):
        y = level + rng.normal(0, 0.01)
        if pump == 0 and y < 2.0:
            pump = 1
        elif pump == 1 and y > 4.0:
            pump = 0
        rows.append((t[k], y, pump, demand[k]))
        level += DT_H * (56.0 * pump - demand[k]) / A
    return pd.DataFrame(rows, columns=["hour", "level", "pump", "true_demand"])

LOG = station_history()
fit_part = LOG.iloc[:-int(24 / DT_H)]
X = np.zeros((len(fit_part) - 1, 25))
X[:, 0] = fit_part.pump.values[:-1]
X[np.arange(len(X)), 1 + (fit_part.hour.values[:-1] % 24).astype(int)] = -1.0
yv = A * np.diff(fit_part.level.values) / DT_H
sol, *_ = np.linalg.lstsq(X, yv, rcond=None)
Q_FIT, D_FIT = sol[0], sol[1:]
print(f"Flow of the pump: {Q_FIT:.1f} m³/h (rated {Q} m³/h)")
print("Hourly demand (m³/h):", np.round(D_FIT, 1))

def predict(level, pumps, hours):
    """The twin from a level over the given pump states and hours."""
    for u, hr in zip(pumps, hours):
        level += DT_H * (Q_FIT * u - D_FIT[int(hr % 24)]) / A
    return level

last = LOG.iloc[-int(24 / DT_H):]
ahead = int(1 / DT_H)                                            # steps in one hour
lv, pu, hr = LOG.level.values, LOG.pump.values, LOG.hour.values
first = len(LOG) - len(last)
pred = [predict(lv[k - ahead], pu[k - ahead:k], hr[k - ahead:k]) for k in range(first, len(LOG))]
open_loop = [predict(lv[first], pu[first:k], hr[first:k]) for k in range(first, len(LOG))]
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
print(f"Validation on the last day: RMSE one hour ahead {100 * rmse(pred, last.level):.1f} cm, "
      f"open loop over the day {100 * rmse(open_loop, last.level):.1f} cm")
RESULTS["fitted flow of the pump (m³/h)"] = round(Q_FIT, 1)
RESULTS["daily demand of the twin (m³)"] = round(float(D_FIT.sum()), 1)
RESULTS["twin: RMSE one hour ahead (cm)"] = round(100 * rmse(pred, last.level), 1)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(last.hour / 24, last.level, lw=0.8, color="#9AA5AE", label="measured")
ax.plot(last.hour / 24, pred, label="twin, one hour ahead")
ax.plot(last.hour / 24, open_loop, "--", lw=1, label="twin, open loop")
ax.set_xlabel("Day")
ax.set_ylabel("Level (m)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. A cheaper pumping schedule

*Lecture: [Week 9, Battery scheduling](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#battery-schedule) and [Week 10, Time series and forecast errors](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#time-series)*

The reservoir stores water as a battery stores energy, so the pump can run when electricity is cheap. The twin forecasts
the demand of tomorrow by its hourly profile. A linear program chooses the share $u_t$ of each hour in which the pump
runs:

$$\min \sum_t \pi_t\,P_{el}\,u_t, \qquad h_t = h_{t-1} + \frac{Q\,u_t - D_t}{A}, \qquad h_{min} \le h_t \le h_{max} ,$$

with $0 \le u_t \le 1$, the price $\pi_t$ and the electric power $P_{el}$ of the pump. The day must end with at least the
level of its start. The cell compares the schedule with the float switches on the same demand. It then runs the
schedule on a demand that differs from the forecast, to show how much margin the limits need.

In [ ]:
# TODO: the tariff, the pump and the limits of your reservoir
PRICE = np.array([0.08] * 6 + [0.16] * 11 + [0.24] * 5 + [0.08] * 2)   # euro/kWh for the hours 0 to 23
P_EL = 22.0                  # electric power of the pump (kW)
H_MIN, H_MAX = 1.5, 4.5      # limits for the schedule (m)
H_START = 3.0                # level at midnight (m)

def schedule(demand, h0=H_START):
    """The linear program of one day: the share of each hour in which the pump runs."""
    n = len(demand)
    L = np.tril(np.ones((n, n)))                                 # cumulative sums
    level_from_u = L * Q_FIT / A                                 # h = h0 + level_from_u @ u - cumsum(D) / A
    base = h0 - np.cumsum(demand) / A
    A_ub = np.vstack([level_from_u, -level_from_u, -level_from_u[-1:]])
    b_ub = np.r_[H_MAX - base, base - H_MIN, base[-1] - h0]
    res = linprog(PRICE * P_EL, A_ub=A_ub, b_ub=b_ub, bounds=[(0, 1)] * n, method="highs")
    return res.x, base + level_from_u @ res.x

def floats(demand, h0=H_START, on=2.0, off=4.0, steps=6):
    """The float switches, simulated in steps of 10 minutes; returns the hourly share of pumping."""
    h, pump, u = h0, 0, np.zeros(len(demand))
    for t in range(len(demand)):
        for _ in range(steps):
            if pump == 0 and h < on:
                pump = 1
            elif pump == 1 and h > off:
                pump = 0
            u[t] += pump / steps
            h += (Q_FIT * pump - demand[t]) / A / steps
    return u

u_lp, h_lp = schedule(D_FIT)
u_fl = floats(D_FIT)
cost_lp, cost_fl = float(PRICE @ (P_EL * u_lp)), float(PRICE @ (P_EL * u_fl))
print(f"Cost of a day: float switches {cost_fl:.2f} euro, schedule {cost_lp:.2f} euro")
actual = D_FIT * np.random.default_rng(3).normal(1.0, 0.1, 24)
h_act = H_START + np.cumsum(Q_FIT * u_lp - actual) / A
print(f"Schedule on a demand 10 percent off the forecast: level between {h_act.min():.2f} and {h_act.max():.2f} m")
RESULTS["cost of a day, float switches (euro)"] = round(cost_fl, 2)
RESULTS["cost of a day, schedule (euro)"] = round(cost_lp, 2)
RESULTS["lowest level with the actual demand (m)"] = round(float(h_act.min()), 2)

fig, ax = plt.subplots(2, 1, figsize=(8, 4.8), sharex=True)
ax[0].bar(range(24), u_lp, color="#1E6FA8", label="share of the hour pumping")
ax[0].step(range(24), PRICE / PRICE.max(), where="mid", color="#C98510", label="price, relative")
ax[1].plot(range(24), h_lp, label="planned level")
ax[1].plot(range(24), h_act, label="level with the actual demand")
ax[1].axhline(H_MIN, color="#B23B3B", ls="--", lw=1)
ax[1].axhline(H_MAX, color="#B23B3B", ls="--", lw=1)
ax[0].set_ylabel("Share, price")
ax[1].set_ylabel("Level (m)")
ax[1].set_xlabel("Hour of the day")
for a in ax:
    a.legend(fontsize=8)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Checking every state of the controller

*Lecture: [Week 14, Verification: Checking every state](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#model-checking)*

The reservoir must never overflow. A model checker proves this for a model of the controller by visiting every reachable
state. The model divides the level into six zones, from 0 for empty to 5 for an overflow, with switches at the zones
High (3) and High-high (4). A state also holds the pump, the radio link and the fault of the switches: None, a stuck High
switch, or both switches stuck by a common cause such as dirt on their float tube. In each step, the controller acts on
the switches that it receives, and the environment moves the level by one zone or keeps it. The link may fail or recover
at any step. Three versions of the controller are checked:

* A stops the pump at the High switch and when the link is lost.
* B also stops the pump at the High-high switch.
* C adds an altitude valve, a mechanical valve that closes the inlet at High-high without any electronics.

A breadth-first search lists all states that a version can reach. If an overflow is among them, its path from the
start is the counterexample.

In [ ]:
# TODO: the model of your controller; a state is (zone, pump, link, fault)
FAULTS = ["none", "High stuck", "High and High-high stuck"]

def readings(zone, fault):
    """What the controller receives: the High and the High-high switch."""
    high = zone >= 3 and fault == 0
    highhigh = zone >= 4 and fault < 2
    return high, highhigh

def successors(state, version):
    zone, pump, link, fault = state
    high, highhigh = readings(zone, fault)
    if not link:
        pump = 0                                                  # no readings: stop the pump
    elif zone <= 1:
        pump = 1                                                  # Low switch: start
    elif high or (version in "BC" and highhigh):
        pump = 0
    inflow = pump and not (version == "C" and zone >= 4)          # the altitude valve closes at High-high
    moves = [1, 0] if inflow else [-1, 0]
    return {(min(5, max(0, zone + m)), pump, new_link, fault) for m in moves for new_link in (0, 1)}

def model_check(version):
    """Breadth-first search over all reachable states; returns their number and the shortest path to an overflow."""
    start = [(2, 0, 1, f) for f in range(3)]
    parent, queue, bad = {s: None for s in start}, deque(start), None
    while queue:
        s = queue.popleft()
        if s[0] == 5 and bad is None:
            bad = s
        for n in successors(s, version):
            if n not in parent:
                parent[n] = s
                queue.append(n)
    path = []
    while bad is not None:
        path.append(bad)
        bad = parent[bad]
    return len(parent), (path[::-1] or None)

for version in "ABC":
    n, path = model_check(version)
    print(f"Version {version}: {n} states reached, " + ("no overflow" if path is None else
          f"overflow after {len(path) - 1} steps with the fault '{FAULTS[path[0][3]]}'"))
    if path:
        print("   path (zone, pump, link, fault):", path)
    RESULTS[f"version {version}: overflow reachable"] = path is not None

The model checker gives a proof only for its model. The model abstracts the level into zones and assumes that the altitude valve itself works. The fault tree of the next section puts numbers on these assumptions.

## 5. Reliability and the fault tree of the overflow

*Lecture: [Week 14, Reliability and redundancy](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#reliability) and [Week 14, Hidden faults and common causes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#hidden-faults) and [Week 14, Fault trees](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#fault-trees)*

An overflow needs a demand and a failed protection. The demand arises when the control does not stop the pump, with the
rate $f_d$ per year. A protection with hidden faults of rate $\lambda$, found by proof tests every $\tau$ years, fails
on demand with the probability $q = \lambda\tau/2$. Two protections of the same kind, 1 out of 2, share a common cause
with the share $\beta$:

$$q_{1oo2} = \beta\,\frac{\lambda\tau}{2} + \frac{\left((1 - \beta)\,\lambda\tau\right)^2}{3} .$$

Diverse protections, such as a switch and a mechanical valve, are taken as independent, so their probabilities multiply.
The rate of overflows is $f = f_d\,q$. The cell first checks $q = \lambda\tau/2$ with a simulation of hidden faults.
The exact share of the time is $1 - (1 - e^{-\lambda\tau})/(\lambda\tau)$, and $\lambda\tau/2$ approximates it from
above for small $\lambda\tau$. The cell then compares the designs by their rate of overflows and their yearly cost, which
includes the expected damage.

In [ ]:
# TODO: the rates and the costs of your system
F_DEMAND = 0.3               # demands per year: the High switch, the controller or the contactor fails to stop the pump
LAM_SW, LAM_VALVE = 0.2, 0.05   # hidden faults per year of a level switch and of the altitude valve
BETA = 0.1                   # share of common causes between two switches
DAMAGE = 20000               # damage of one overflow (euro)
COST_TEST = 200              # cost of one proof test (euro)
COST_VALVE = 400             # yearly cost of the altitude valve, with its installation spread over its life (euro)
TARGET = 0.01                # largest acceptable overflows per year

# A check of q = lambda tau / 2: the share of the time in which a hidden fault waits for the next proof test
rng = np.random.default_rng(16)
tau = 1.0
fail = rng.exponential(1 / LAM_SW, 200000)
q_sim = float(np.mean(np.clip(tau - fail, 0, None)) / tau)
q_exact = 1 - (1 - math.exp(-LAM_SW * tau)) / (LAM_SW * tau)
print(f"Simulated q = {q_sim:.4f}, exact {q_exact:.4f}, approximation lambda tau / 2 = {LAM_SW * tau / 2:.4f}")

def q1(lam, tau):
    return lam * tau / 2

def q1oo2(lam, tau, beta=BETA):
    return beta * lam * tau / 2 + ((1 - beta) * lam * tau) ** 2 / 3

DESIGNS = [  # name, probability of failure on demand, proof tests per year, valve
    ("A: no protection", 1.0, 0, False),
    ("B: High-high switch, test every year", q1(LAM_SW, 1.0), 1, False),
    ("B: High-high switch, test every 3 months", q1(LAM_SW, 0.25), 4, False),
    ("B2: two High-high switches, test every year", q1oo2(LAM_SW, 1.0), 2, False),
    ("C: altitude valve, test every year", q1(LAM_VALVE, 1.0), 1, True),
    ("B + C: switch and valve, test every year", q1(LAM_SW, 1.0) * q1(LAM_VALVE, 1.0), 2, True),
]
rows = []
for name, q, tests, valve in DESIGNS:
    f = F_DEMAND * q
    cost = f * DAMAGE + tests * COST_TEST + (COST_VALVE if valve else 0)
    rows.append({"design": name, "q": round(q, 5), "overflows per year": round(f, 5),
                 "meets the target": f <= TARGET, "yearly cost (euro)": round(cost, 0)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
ok = table[table["meets the target"]]
best = ok.loc[ok["yearly cost (euro)"].idxmin()]
print(f"Cheapest design that meets the target: {best['design']}, {best['yearly cost (euro)']:.0f} euro per year")
RESULTS["simulated q of a switch"] = round(q_sim, 4)
RESULTS["chosen design"] = best["design"]
RESULTS["overflows per year of the chosen design"] = float(best["overflows per year"])
RESULTS["yearly cost of the chosen design (euro)"] = float(best["yearly cost (euro)"])

Write the safety case of the report as the claim *the reservoir does not overflow*, supported by evidence. The evidence here is the model check, the fault tree with its numbers and the timing of Section 1. Name the assumption that you trust least, for example the independence of the switch and the valve.

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>